# Setup (do this once, before class)

1. Go to your home directory and create a working directory: `mkdir isa460`
2. Change into it: `cd isa460`
3. Clone the textbook's [code](https://github.com/jonesberg/DataAnalysisWithPythonAndPySpark) and [data](https://github.com/jonesberg/DataAnalysisWithPythonAndPySpark-Data)
4. Rename `DataAnalysisWithPythonAndPySpark-Data` to `data`
5. Clone the [ISA460 course repo](https://github.com/Suhong88/isa460_fall2026) into the same directory


# My First PySpark Program

**Question: what are the most common words in the English language?**
(using Jane Austen's *Pride and Prejudice* as our sample of English)

| Step | What it does |
|------|--------------|
| 1. **Read** | Load the input data (a plain text file) |
| 2. **Token** | Split each line into words |
| 3. **Clean** | Lowercase, strip punctuation, drop empties |
| 4. **Count** | Count how often each word appears |
| 5. **Answer** | Return the top 10 |

---
**How this notebook works.** Cells with `___` are for you to fill in. Each one
says what correct output looks like, so you can check yourself. Ask if you are
stuck for more than a couple of minutes — the point is the idea, not the typing.


## Step 1 — Read

The setup below is given. **Change `account` to your own username**, then run it.

In [1]:
from pyspark.sql import SparkSession

# CHANGE THIS to your Bryant username
account = 'sli'

# where the textbook data lives on the cluster
data_path = '/net/clusterhn/home/' + account + '/isa460/data/'

spark = (SparkSession.builder.appName("My First Spark Program")
         .config("spark.port.maxRetries", "100")
         .getOrCreate())

# quieten Spark's logging (default is WARN)
spark.sparkContext.setLogLevel('ERROR')

book = spark.read.text(data_path + "gutenberg_books/1342-0.txt")

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/09/07 08:42:50 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/09/07 08:42:51 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.


### Look at what you loaded

Before transforming anything, always look at it. Two questions to answer:
what are the columns, and what does a row look like?

In [ ]:
# Print the structure (schema) of the data
book.___()

# Check: one column called "value", of type string

In [ ]:
# Show the first 10 rows, truncating long lines to 50 characters
book.___(___, truncate=50)

# Check: the opening lines of Pride and Prejudice, one line of text per row


## Step 2 — Tokenise: from a sentence to a list of words

### Four ways to name a column

All four of these do the same thing. Use whichever reads best:

```python
from pyspark.sql.functions import col

book.select(book.value)
book.select(book["value"])
book.select(col("value"))
book.select("value")
```

### Renaming a column
Use `.alias()` inside a `select()`, or `.withColumnRenamed()` on the DataFrame.

### Split each line into a list of words

`split(column, delimiter)` turns one string into an array of strings.

In [ ]:
from pyspark.sql.functions import split

lines = book.select(split(book.value, ___).alias("line"))
lines.show(5)

# Check: each row holds a LIST like [The, Project, Gutenberg, EBook, ...]
# Note the square brackets - this is still one row per line of the book.

### Explode the list into rows

`split` gave us one row per *line*, each holding a list. We want one row per
*word*. `explode` turns each element of a list into its own row.

In [ ]:
from pyspark.sql.functions import explode, col

words = lines.select(___(col("line")).alias("word"))
words.show(15)

# Check: one word per row now, no square brackets.


## Step 3 — Clean: lowercase, then strip punctuation

In [ ]:
from pyspark.sql.functions import lower

words_lower = words.select(___(col("word")).alias("word_lower"))
words_lower.show()

# Check: "The" has become "the". Punctuation is still attached.


### Remove anything that is not a letter

`regexp_extract(column, pattern, group)` keeps only the part of the string that
matches a regular expression.

- `[a-z]+` means "one or more lowercase letters"
- group `0` means "the whole match"

[Regular expression reference](https://docs.python.org/3/howto/regex.html)

In [ ]:
from pyspark.sql.functions import regexp_extract

words_clean = words_lower.select(
    regexp_extract(col("word_lower"), ___, 0).alias("word")
)
words_clean.show()

# Check: "prejudice," has become "prejudice". Some rows are now EMPTY - that is
# expected, and we deal with it next.


In [ ]:
# Uncomment to read the documentation for any function
#help(regexp_extract)


### Drop the empty rows

Cleaning produced empty strings where a token had no letters at all. Filter
them out. `.filter()` and `.where()` are the same method — use either.

In [ ]:
words_nonull = words_clean.filter(col("word") ___ "")
words_nonull.show()

# Check: no blank rows remain.


## Step 4 — Count

`groupBy` collects rows that share a value; the aggregation that follows says
what to compute for each group.

![GroupedData](https://raw.githubusercontent.com/Suhong88/ISA460_Fall2023/main/images/3.1.A.png)

In [ ]:
results = words_nonull.___(col('word')).___()
results.show()

# Check: two columns, "word" and "count". The order looks random - that is
# normal, Spark has not been asked to sort anything yet.


### Your turn: how long are English words?

Count how many words have 1 letter, 2 letters, and so on.

In [ ]:
from pyspark.sql.functions import length

(words_nonull
    .withColumn('length', ___(col('word')))
    .___('length')
    .___()
    .show())

# Check: 3-letter words are the most common at 28,831, then 2-letter at
# 23,856, then 4-letter at 22,213. Note the rows come back unsorted.


## Step 5 — Answer: order the results

Two equivalent ways to sort descending.

Fill in both.

In [ ]:
results.orderBy("count", ascending=___).show(10)


In [ ]:
results.orderBy(col("count").___()).show(10)

# Check: both give the same top 10
#   the 4496 | to 4235 | of 3719 | and 3602 | her 2223
#   i 2052 | a 1997 | in 1920 | was 1844 | she 1703
# Almost all function words - worth a thought about what that says about
# measuring "importance" by frequency.


## Writing results out

Spark writes a *directory*, not a single file — one part-file per partition.
That is a feature, not a bug: it is how a cluster writes in parallel.

In [ ]:
path = '/net/clusterhn/home/' + account + '/isa460/'

results.write.csv(path + 'simple_count.csv')

# Check: look in your isa460 directory. simple_count.csv is a FOLDER.
# Re-running this cell will fail because the folder exists - see the
# .mode('overwrite') option used later.


## Putting it all together

Now write the whole pipeline yourself, in one cell, from a clean start. This is
the part worth struggling with — everything above is available to scroll back to.

Two things differ from what we built above:
- `.coalesce(1)` gathers the result into a single partition, so you get one CSV
- `.mode('overwrite')` lets you re-run the cell without deleting the folder first


In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, explode, lower, regexp_extract, split

account = 'sli'          # your username
data_path = '/net/clusterhn/home/' + account + '/isa460/data/'

spark = (SparkSession.builder
         .appName("Analyzing the vocabulary of Pride and Prejudice.")
         .config("spark.port.maxRetries", "100")
         .getOrCreate())

book = spark.read.text(data_path + "gutenberg_books/1342-0.txt")

lines = ___

words = ___

words_lower = ___

words_clean = ___

words_nonull = ___

results = ___

results.orderBy("count", ascending=False).show(10)

path = '/net/clusterhn/home/' + account + '/isa460/'
results.coalesce(1).write.mode('overwrite').csv(path + "simple_count_single_partition.csv")


## Simplifying with method chaining

Each step above created a DataFrame we never used again — `lines`, `words`,
`words_lower`. Chaining drops those intermediate names and reads as one
pipeline. Same result, less clutter.

![Method Chaining 3.3](https://raw.githubusercontent.com/Suhong88/ISA460_Fall2023/main/images/3.3.png)

Note `import pyspark.sql.functions as F` — the usual convention, so it is
obvious which functions come from Spark rather than plain Python.

In [ ]:
import pyspark.sql.functions as F

results = (
    spark.read.text(data_path + "gutenberg_books/1342-0.txt")
    .select(F.split(F.col("value"), " ").alias("line"))
    .select(___)
    .select(___)
    .select(___)
    .where(___)
    .groupby("word")
    .count()
)

results.show()

# Check: same numbers as before. If they differ, compare your chain with the
# step-by-step version above - one of the stages is doing something different.


## Running it outside a notebook

A notebook is for exploring. Production Spark jobs are submitted as scripts.

1. Save your word count as a Python script, `word_count.py`
2. Open a terminal
3. Activate the environment: `conda activate BigDataEnv`
4. Change to the directory holding the script
5. Run it: `spark-submit word_count.py`

Note that a script needs `spark.stop()` at the end, and that `.show()` prints to
the terminal rather than into a notebook cell.

---
## On your own

Try these with the same book, or another one from `gutenberg_books/`.

1. **The top 10 excluding stop words.** Filter out the ten most common words
   (`the, to, of, and, her, i, a, in, was, she`) and look again. Do the words
   that remain tell you what the book is about? Hint: `~col("word").isin([...])`

2. **Longest words.** Which words in the book are longest? Watch out for
   hyphenated tokens that `split(" ")` kept together.

3. **Compare two books.** Run the pipeline over a second file in
   `gutenberg_books/` and compare the top 10. How much overlap is there — and
   what does that suggest about which words carry meaning?

4. **A better tokeniser.** `split(" ")` breaks on spaces only, so `end.` and
   `end` are different tokens until `regexp_extract` fixes it. What happens to
   words with apostrophes, like `don't`? Try the pattern `[a-z']*` used in the
   textbook and see what changes.
